In [2]:
# ===== Imports =====
import pandas as pd
import requests
import random
import time
from fake_useragent import UserAgent
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

In [ ]:
# ===== Downloading ids of profiles =====
ua = UserAgent()
profiles_ids = []
problem_pages = []
basic_url = 'https://repetit.ru/repetitors/matematika/?page={:d}'

# The total amount of profiles is 6600-6700, 10 per page
# So, totally we need to process nearly 660-670 pages 
# Here is a page iterations process with a small margin 
for i in range(1, 700):
    try:
        random_ua = ua.random
        new_header = {'User-Agent': random_ua}
        page = requests.get(basic_url.format(i), headers = new_header)
        new_ids = re.findall(r'<article class=\"teacher-card\" data-teacher-id=\"(\d+)\"', page.text)
        profiles_ids.extend(new_ids)
        print(f'Со страницы {i} получены новые id: \n {new_ids}✔️')
        time.sleep(random.uniform(1, 2))
    except:
        print(f"Something wrang with page {i}🚨")
        problem_pages.append(i)

ids_set = set(profiles_ids)
with open('profiles_ids.txt', 'x', encoding='utf-8') as file:
    for id in ids_set:
        file.write(id + '\n')

In [3]:
# ===== Functions for parsing =====
def safe_text(element):
    if element is None:
        return ''
    text = element.string
    return text.strip() if text else ''

def profile_parsing(soup):

    name = safe_text(soup.find('h1', attrs={"itemprop": "name"}))
    age = safe_text(soup.find('span', class_ = 'intro__age-value'))
    status = safe_text(soup.find('span', class_ = 'intro__status-value'))
    experience = safe_text(soup.find('span', class_ = 'intro__experience-value'))
    rating = safe_text(soup.find('p', class_ = 'intro__star-rating'))
    n_reviews = safe_text(soup.find('p', class_ = 'intro__reviews-count info-point info-point_column'))
    n_last_month_views = soup.find_all('span', class_ = 'rating__text_bold')[1].text.strip() or ''
    last_activity_date = soup.find_all('span', class_ = 'rating__text_bold')[2].text.strip() or ''
    n_students_placed = soup.find_all('span', class_ = 'rating__text_bold')[3].text.strip() or ''
    description = safe_text(soup.find('span', id = 'description-paragraph'))

    subject_card = soup.find(
    lambda tag: tag.name == "article" 
    and tag.get("class") == ["subject"] 
    and tag.find("h3", string=lambda s: s and "Математика" in s)
    )
    if subject_card:
        online_math = subject_card.find(
        lambda tag: tag.name == "li" 
        and tag.get("class") == ["prices__list-item"]
        and tag.find("span", class_ = 'price__title', string=lambda s: s and "Онлайн" in s)
        )
        if online_math:
            money = safe_text(online_math.find('span', class_ = 'price__value'))
            duration = safe_text(online_math.find('span', class_ = 'price__per-hour'))
            price = money + duration
        else:
            price = ''
    else:
        price = ''

    return [
        name, age, status, experience, rating, n_reviews,
        n_last_month_views, last_activity_date,
        n_students_placed, description, price
    ]

In [ ]:
# ===== Profiles parsing =====
profiles_info_df = pd.DataFrame(columns=[
    'Name', 'Age', 'Status', 'Experience', 'Rating', 'N_reviews',
    'N_last_month_views', 'Last_activity_date', 'N_students_placed',
    'Description', 'Price'
])

mistakes_ids = []
no_mistake_count = 0
mistakes_count = 0

rep_url = 'https://repetit.ru/repetitor.aspx?id={:d}'
ua = UserAgent()

with open('profiles_ids.txt', 'r', encoding='utf-8') as file:
    profile_ids_list = [int(line.strip()) for line in file]

with tqdm(profile_ids_list, desc="Parsing", unit="profile") as pbar:
    n_profile = 1
    for id in pbar:
        try:
            random_ua = ua.random
            new_header = {'User-Agent': random_ua}
            new_repetitor = requests.get(rep_url.format(id), headers = new_header, timeout = 30)
            html_content = new_repetitor.text
            soup = BeautifulSoup(html_content, "html.parser")
            
            profiles_info_df.loc[n_profile] = profile_parsing(soup)

            # print(f'Profile {id} successfully parsed ✅')
            no_mistake_count += 1
            n_profile += 1
        except:
            # print(f'Something wrang with profile {id}❌')
            mistakes_ids.append(id)
            mistakes_count += 1
            continue
        
profiles_info_df.to_csv('profiles.csv', index=False)
print(f'''
{no_mistake_count} profiles successfully parsed ✅
{mistakes_count} profiles with mistakes ❌
''')